In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

In [58]:
df = pd.read_csv("powerplat_data.csv")
df.shape

(9568, 5)

In [59]:
X = df.drop("PE", axis = 1)
y = df["PE"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [60]:
#Converting the data into Tensors(PyTorch core data structures)
X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype = torch.float32).view(-1,1)#y_train is not scaled and still a numpy array so we use values for that
#.view changes the dimensions - [1,2,3,4] -> [[1], [2], [3], [4]], -1 means detect automatically
X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype = torch.float32).view(-1,1)

#all the tensors will be stored in RAM of our computer

In [61]:
# coverting in DataSets and DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32)

# Deep learning

In [62]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        self.model = nn.Sequential(
            # 1st layer 
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            # 2nd layer
            nn.Linear(6,6),
            nn.ReLU(),

            # output layer
            nn.Linear(6,1)
        )
    def forward(self, x):
        return self.model(x)

In [63]:
model = ANN()

criteria = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [64]:
# training the model

epochs = 100
for epoch in range(epochs):
    model.train()
    running_loss = 0
    for xb,yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)
    print(f"epoch is {epoch + 1} and loss is {train_loss}")

epoch is 1 and loss is 205815.95182291666
epoch is 2 and loss is 192483.09869791666
epoch is 3 and loss is 147425.4093424479
epoch is 4 and loss is 84307.96917317709
epoch is 5 and loss is 40801.963541666664
epoch is 6 and loss is 21544.781339518227
epoch is 7 and loss is 13636.23087158203
epoch is 8 and loss is 9834.383858235677
epoch is 9 and loss is 7219.293663533529
epoch is 10 and loss is 5272.309560139974
epoch is 11 and loss is 3822.1067143758137
epoch is 12 and loss is 2763.8883229573566
epoch is 13 and loss is 1986.7210845947266
epoch is 14 and loss is 1437.167902882894
epoch is 15 and loss is 1040.2531430562337
epoch is 16 and loss is 744.0222915649414
epoch is 17 and loss is 524.8910258928935
epoch is 18 and loss is 368.57671721776325
epoch is 19 and loss is 257.06379823684694
epoch is 20 and loss is 181.69843999544779
epoch is 21 and loss is 131.65582466920216
epoch is 22 and loss is 101.17066318194071
epoch is 23 and loss is 74.95199428399404
epoch is 24 and loss is 59.400

In [69]:
# evaluation
model.eval()
with torch.no_grad():
    train_preds = model(X_train_tensor)
    test_preds = model(X_test_tensor)

    train_mse_loss = criteria(train_preds, y_train_tensor)
    test_mse_loss = criteria(test_preds, y_test_tensor)
print(f"Training mse {train_mse_loss.item()}")
print(f"Testing mse {test_mse_loss.item()}")

Training mse 20.366125106811523
Testing mse 18.677553176879883


In [70]:
from sklearn.metrics import r2_score

In [71]:
print(f"r2_score is {r2_score(train_preds, y_train_tensor)}")

r2_score is 0.9249758720397949
